NoteBook 03 - Feature Engineering and Target Creation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import glob


Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation"
FINAL_PROJECT_PATH = PROJECT_PATH + "/Final_Project"

CLEAN_DATA_PATH = FINAL_PROJECT_PATH + "/02_Data/Cleaned"
FEATURES_PATH = FINAL_PROJECT_PATH + "/02_Data/Processed/Features"
SPLITS_PATH = FINAL_PROJECT_PATH + "/02_Data/Splits"
RESULTS_PATH = FINAL_PROJECT_PATH + "/04_Results"

os.makedirs(FEATURES_PATH, exist_ok=True)
os.makedirs(SPLITS_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

#Target/labeling configuration
HORIZON_BARS = 4          # predicting the return 4 bars ahead (~1 hour on 15-min data)
VOLATILITY_WINDOW = 20    # bars used to measure each stock's own recent volatility
THRESHOLD_MULTIPLIER = 0.5  # Buy/Sell boundary = +/- (multiplier x rolling volatility)
TRAIN_FRACTION = 0.8      # first 80% of each stock's timeline = train, last 20% = test

print("Cleaned data folder:", CLEAN_DATA_PATH)


Cleaned data folder: /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/02_Data/Cleaned


In [ ]:
clean_files = glob.glob(os.path.join(CLEAN_DATA_PATH, "*.parquet"))
stock_data = {}

for file_path in clean_files:
    stock_name = os.path.basename(file_path).replace(".parquet", "")
    df = pd.read_parquet(file_path)
    df = df.sort_values("datetime").reset_index(drop=True)  # belt-and-braces
    stock_data[stock_name] = df

print("Stocks loaded:", list(stock_data.keys()))
print("Example shape:", list(stock_data.values())[0].shape)


Stocks loaded: ['Eicher_Motors', 'Mahindra_and_Mahindra', 'HDFCB', 'ICICIBC', 'ONGC', 'Reliance', 'Hindustan_Unilever', 'ITC', 'INFOSYS', 'TCS']
Example shape: (72495, 7)


In [ ]:
def add_returns(df):
    df["return_1"] = df["close"].pct_change()
    df["log_return_1"] = np.log(df["close"] / df["close"].shift(1))
    return df


def add_moving_averages(df, windows=(5, 10, 20, 50)):
    for w in windows:
        df[f"sma_{w}"] = df["close"].rolling(window=w, min_periods=w).mean()
        df[f"ema_{w}"] = df["close"].ewm(span=w, adjust=False, min_periods=w).mean()
    return df


def add_rsi(df, window=14):
    # Wilder's RSI: average gain vs average loss over the window, both computed
    # with an exponential moving average that only ever looks backward.
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()

    rs = avg_gain / avg_loss
    df["rsi_14"] = 100 - (100 / (1 + rs))
    return df


def add_macd(df, fast=12, slow=26, signal=9):
    ema_fast = df["close"].ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = df["close"].ewm(span=slow, adjust=False, min_periods=slow).mean()

    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=signal, adjust=False, min_periods=signal).mean()
    df["macd_hist"] = df["macd_line"] - df["macd_signal"]
    return df


def add_bollinger_bands(df, window=20, num_std=2):
    sma = df["close"].rolling(window=window, min_periods=window).mean()
    std = df["close"].rolling(window=window, min_periods=window).std()

    df["bb_mid"] = sma
    df["bb_upper"] = sma + num_std * std
    df["bb_lower"] = sma - num_std * std
    # %B: where is price sitting within the bands (0 = at lower band, 1 = at upper band)
    df["bb_percent_b"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])
    # bandwidth: how wide the bands are relative to price - a volatility-squeeze indicator
    df["bb_bandwidth"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_mid"]
    return df


def add_volatility_and_volume_features(df, window=20):
    df["volatility_20"] = df["return_1"].rolling(window=window, min_periods=window).std()
    df["volume_avg_20"] = df["volume"].rolling(window=window, min_periods=window).mean()
    df["volume_relative"] = df["volume"] / df["volume_avg_20"]
    return df


Buy/Sell/Hold labels

In [ ]:
def add_target(df, horizon=HORIZON_BARS, vol_window=VOLATILITY_WINDOW, threshold_mult=THRESHOLD_MULTIPLIER):
    # future_return DELIBERATELY uses shift(-horizon) - it looks FORWARD.
    # Trying to predict the label, not a feature. only used as y.
    df["future_return"] = df["close"].shift(-horizon) / df["close"] - 1

    # The threshold, by contrast, uses only PAST data up to and including the current
    # row (rolling, not centered) - this is what makes the labeling itself lookahead-safe.
    rolling_vol = df["return_1"].rolling(window=vol_window, min_periods=vol_window).std()
    df["target_threshold"] = rolling_vol * threshold_mult

    conditions = [
        df["future_return"] > df["target_threshold"],
        df["future_return"] < -df["target_threshold"],
    ]
    choices = ["Buy", "Sell"]
    df["target_signal"] = np.select(conditions, choices, default="Hold")

    return df


In [ ]:
FEATURE_COLUMNS = []

processed_data = {}

for stock_name, df in stock_data.items():
    df = df.copy()

    df = add_returns(df)
    df = add_moving_averages(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_bollinger_bands(df)
    df = add_volatility_and_volume_features(df)
    df = add_target(df)

    processed_data[stock_name] = df
    print(f"{stock_name:25s} -> {df.shape[1]} columns, {len(df)} rows (before dropping warm-up/tail NaNs)")

# Records which columns are model INPUT features - explicitly excludes datetime, the raw price/volume columns (not used directly), and everything target-related.
sample_df = list(processed_data.values())[0]
EXCLUDE_FROM_FEATURES = {
    "datetime", "open", "high", "low", "close", "volume", "tick_count",
    "future_return", "target_threshold", "target_signal",
}
FEATURE_COLUMNS = [c for c in sample_df.columns if c not in EXCLUDE_FROM_FEATURES]

print("\nFeature columns that will be used as model inputs:")
for c in FEATURE_COLUMNS:
    print(" ", c)


Eicher_Motors             -> 32 columns, 72495 rows (before dropping warm-up/tail NaNs)
Mahindra_and_Mahindra     -> 32 columns, 72991 rows (before dropping warm-up/tail NaNs)
HDFCB                     -> 32 columns, 72984 rows (before dropping warm-up/tail NaNs)
ICICIBC                   -> 32 columns, 73084 rows (before dropping warm-up/tail NaNs)
ONGC                      -> 32 columns, 73095 rows (before dropping warm-up/tail NaNs)
Reliance                  -> 32 columns, 73094 rows (before dropping warm-up/tail NaNs)
Hindustan_Unilever        -> 32 columns, 72939 rows (before dropping warm-up/tail NaNs)
ITC                       -> 32 columns, 73082 rows (before dropping warm-up/tail NaNs)
INFOSYS                   -> 32 columns, 73088 rows (before dropping warm-up/tail NaNs)
TCS                       -> 32 columns, 73061 rows (before dropping warm-up/tail NaNs)

Feature columns that will be used as model inputs:
  return_1
  log_return_1
  sma_5
  ema_5
  sma_10
  ema_10
  sma_20

Dropping the NaN rows

In [ ]:
final_data = {}

for stock_name, df in processed_data.items():
    before = len(df)
    df_clean = df.dropna(subset=FEATURE_COLUMNS + ["future_return", "target_threshold"]).reset_index(drop=True)
    final_data[stock_name] = df_clean
    print(f"{stock_name:25s} {before} -> {len(df_clean)} rows ({before - len(df_clean)} dropped as warm-up/tail)")


Eicher_Motors             72495 -> 72442 rows (53 dropped as warm-up/tail)
Mahindra_and_Mahindra     72991 -> 72938 rows (53 dropped as warm-up/tail)
HDFCB                     72984 -> 72931 rows (53 dropped as warm-up/tail)
ICICIBC                   73084 -> 73031 rows (53 dropped as warm-up/tail)
ONGC                      73095 -> 73042 rows (53 dropped as warm-up/tail)
Reliance                  73094 -> 73041 rows (53 dropped as warm-up/tail)
Hindustan_Unilever        72939 -> 72886 rows (53 dropped as warm-up/tail)
ITC                       73082 -> 73029 rows (53 dropped as warm-up/tail)
INFOSYS                   73088 -> 73035 rows (53 dropped as warm-up/tail)
TCS                       73061 -> 73008 rows (53 dropped as warm-up/tail)


In [ ]:
balance_rows = []

for stock_name, df in final_data.items():
    counts = df["target_signal"].value_counts()
    balance_rows.append({
        "Stock": stock_name,
        "Rows": len(df),
        "Buy": counts.get("Buy", 0),
        "Hold": counts.get("Hold", 0),
        "Sell": counts.get("Sell", 0),
        "Buy_%": round(counts.get("Buy", 0) / len(df) * 100, 1),
        "Hold_%": round(counts.get("Hold", 0) / len(df) * 100, 1),
        "Sell_%": round(counts.get("Sell", 0) / len(df) * 100, 1),
    })

balance_df = pd.DataFrame(balance_rows)
balance_df


,Stock,Rows,Buy,Hold,Sell,Buy_%,Hold_%,Sell_%
0,Eicher_Motors,72442,25872,20720,25850,35.7,28.6,35.7
1,Mahindra_and_Mahindra,72938,26416,20034,26488,36.2,27.5,36.3
2,HDFCB,72931,27083,19938,25910,37.1,27.3,35.5
3,ICICIBC,73031,26919,19236,26876,36.9,26.3,36.8
4,ONGC,73042,25654,21080,26308,35.1,28.9,36.0
5,Reliance,73041,26376,19631,27034,36.1,26.9,37.0
6,Hindustan_Unilever,72886,26745,20105,26036,36.7,27.6,35.7
7,ITC,73029,25951,20158,26920,35.5,27.6,36.9
8,INFOSYS,73035,26077,21101,25857,35.7,28.9,35.4
9,TCS,73008,26184,20793,26031,35.9,28.5,35.7


Training and Testing Split

In [ ]:
split_summary_rows = []

for stock_name, df in final_data.items():
    split_idx = int(len(df) * TRAIN_FRACTION)

    train_df = df.iloc[:split_idx].reset_index(drop=True)
    test_df = df.iloc[split_idx:].reset_index(drop=True)

    safe_name = stock_name.replace(" ", "_").replace("&", "and")

    train_df.to_parquet(os.path.join(SPLITS_PATH, f"{safe_name}_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(SPLITS_PATH, f"{safe_name}_test.parquet"), index=False)

    df.to_parquet(os.path.join(FEATURES_PATH, f"{safe_name}_features.parquet"), index=False)

    split_summary_rows.append({
        "Stock": stock_name,
        "Total_Rows": len(df),
        "Train_Rows": len(train_df),
        "Train_Start": train_df["datetime"].min(),
        "Train_End": train_df["datetime"].max(),
        "Test_Rows": len(test_df),
        "Test_Start": test_df["datetime"].min(),
        "Test_End": test_df["datetime"].max(),
    })

split_summary_df = pd.DataFrame(split_summary_rows)
split_summary_df


,Stock,Total_Rows,Train_Rows,Train_Start,Train_End,Test_Rows,Test_Start,Test_End
0,Eicher_Motors,72442,57953,2016-01-04 12:45:00,2024-06-20 13:45:00,14489,2024-06-20 14:00:00,2026-07-31 13:15:00
1,Mahindra_and_Mahindra,72938,58350,2016-01-04 12:45:00,2024-06-21 14:15:00,14588,2024-06-24 07:30:00,2026-07-31 13:15:00
2,HDFCB,72931,58344,2016-01-04 12:45:00,2024-06-24 09:15:00,14587,2024-06-24 09:30:00,2026-07-31 13:15:00
3,ICICIBC,73031,58424,2016-01-04 12:45:00,2024-06-21 10:15:00,14607,2024-06-21 10:30:00,2026-07-31 13:15:00
4,ONGC,73042,58433,2016-01-04 12:45:00,2024-06-21 10:45:00,14609,2024-06-21 11:00:00,2026-07-31 13:15:00
5,Reliance,73041,58432,2016-01-04 12:45:00,2024-06-21 10:45:00,14609,2024-06-21 11:00:00,2026-07-31 13:15:00
6,Hindustan_Unilever,72886,58308,2016-01-04 12:45:00,2024-06-24 07:45:00,14578,2024-06-24 08:00:00,2026-07-31 13:15:00
7,ITC,73029,58423,2016-01-04 12:45:00,2024-06-21 10:30:00,14606,2024-06-21 10:45:00,2026-07-31 13:15:00
8,INFOSYS,73035,58428,2016-01-04 12:45:00,2024-06-21 11:15:00,14607,2024-06-21 11:30:00,2026-07-31 13:15:00
9,TCS,73008,58406,2016-01-04 12:45:00,2024-06-21 12:00:00,14602,2024-06-21 12:15:00,2026-07-31 13:15:00


In [ ]:
summary_path = os.path.join(RESULTS_PATH, "Feature_Target_Summary.csv")
split_summary_df.to_csv(summary_path, index=False)
balance_df.to_csv(os.path.join(RESULTS_PATH, "Class_Balance_Summary.csv"), index=False)

print("Saved:", summary_path)
print("Feature columns used (save this list, you'll need it in the next notebook):")
print(FEATURE_COLUMNS)


Saved: /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/04_Results/Feature_Target_Summary.csv
Feature columns used (save this list, you'll need it in the next notebook):
['return_1', 'log_return_1', 'sma_5', 'ema_5', 'sma_10', 'ema_10', 'sma_20', 'ema_20', 'sma_50', 'ema_50', 'rsi_14', 'macd_line', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_percent_b', 'bb_bandwidth', 'volatility_20', 'volume_avg_20', 'volume_relative']
